# Lab 01 — Biological Neurons
## Membrane as Capacitor · Nernst Equilibrium · Goldman-Hodgkin-Katz Equation

**Computational Neuroscience — Student Notebook**

---

**Learning objectives**

By the end of this lab you should be able to:
1. Model the neuronal membrane as a leaky RC circuit and simulate its voltage response.
2. Calculate Nernst equilibrium potentials for key ions from first principles.
3. Apply the Goldman-Hodgkin-Katz (GHK) equation to predict the resting membrane potential.
4. Describe how a brief synaptic current pulse is integrated across the membrane time constant.

**How to use this notebook**

- Read the theory cells carefully before attempting each exercise.
- Cells marked `# --- YOUR CODE HERE ---` / `# --- END ---` are for you to complete.
- Short-answer cells (marked **[Short answer]**) should be filled in with text.
- Do **not** change function signatures — the test assertions rely on them.

---

## Setup

Run this cell first. It imports all libraries used throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed
from IPython.display import display

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# Physical constants (SI)
R = 8.314    # J / (mol·K)  — gas constant
F = 96485.0  # C / mol       — Faraday constant

# Typical mammalian ion concentrations (mM)
CONC = dict(
    K_in  = 140.0,  K_out  =   5.0,
    Na_in =  15.0,  Na_out = 145.0,
    Cl_in =  10.0,  Cl_out = 110.0,
    Ca_in =   0.1e-3, Ca_out = 2.0,   # mM (intracellular Ca²⁺ is very low)
)

print('Setup complete.')

---
## Part 1 — Membrane as an Imperfect Capacitor

### 1.1  The RC membrane model

The lipid bilayer acts as a **capacitor**: two conducting leaflets (intra- and extracellular electrolyte) separated by a thin insulating core (~3 nm). The specific membrane capacitance is remarkably constant across cell types:

$$C_m \approx 1\;\mu\text{F/cm}^2$$

However, the membrane is **leaky** — ion channels create resistive pathways. In the simplest passive model we lump all leak channels into a single conductance $g_L$ (units: mS/cm²) with reversal potential $E_L$ (units: mV):

$$C_m \frac{dV}{dt} = -g_L\,(V - E_L) + I_{\text{ext}}$$

where $I_{\text{ext}}$ (µA/cm²) is an externally injected current.

**Analytical solution.** Defining the steady-state voltage $V_\infty = E_L + I_{\text{ext}}/g_L$ and the **membrane time constant** $\tau_m = C_m / g_L$, the solution is:

$$V(t) = V_\infty + \bigl(V(0) - V_\infty\bigr)\,e^{-t/\tau_m}$$

$\tau_m$ sets how fast the membrane responds to inputs — typically 10–30 ms for cortical neurons.

### 1.2  Euler integration

We will approximate the ODE numerically using the **forward Euler** method. For a time step $\Delta t$:

$$V(t + \Delta t) \approx V(t) + \Delta t \cdot \frac{dV}{dt}\bigg|_t$$

Choose $\Delta t \ll \tau_m$ for accuracy (we use $\Delta t = 0.1$ ms).

### Exercise 1.1 — Implement the passive membrane simulation

Complete the function below using the Euler update rule derived above.

In [ ]:
def simulate_passive_membrane(I_ext, T=200.0, dt=0.1,
                               Cm=1.0, gL=0.1, EL=-70.0, V0=-70.0):
    """
    Simulate a passive (leaky) membrane using the Euler method.

    Parameters
    ----------
    I_ext : float   External current (µA/cm²)
    T     : float   Total simulation time (ms)
    dt    : float   Time step (ms)
    Cm    : float   Membrane capacitance (µF/cm²)
    gL    : float   Leak conductance (mS/cm²)
    EL    : float   Leak reversal potential (mV)
    V0    : float   Initial membrane voltage (mV)

    Returns
    -------
    t : np.ndarray  Time vector (ms)
    V : np.ndarray  Voltage vector (mV)
    """
    # --- YOUR CODE HERE ---
    # Hint 1: Build the time array with np.arange(0, T, dt)
    # Hint 2: Allocate a voltage array of the same length, set V[0] = V0
    # Hint 3: Loop from index 1 onward; at each step compute dV/dt from
    #         the ODE above, then update:  V[i] = V[i-1] + dt * dV_dt
    # Hint 4: Note that Cm, gL are in compatible units when T and dt are in ms
    #         (because µF/cm² ÷ mS/cm² = µF/mS = 10⁻⁶ F / 10⁻³ S = ms)
    pass
    # --- END ---


# --- Quick plot (runs after you implement the function) ---
t, V = simulate_passive_membrane(I_ext=1.0)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t, V, color='steelblue', lw=1.8)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Membrane voltage (mV)')
ax.set_title('Passive membrane — $I_{ext}$ = 1 µA/cm²')
plt.tight_layout()
plt.show()

### Exercise 1.2 — Time constant and steady-state voltage

Recall the analytical expressions:

$$\tau_m = \frac{C_m}{g_L}, \qquad V_\infty = E_L + \frac{I_{\text{ext}}}{g_L}$$

Fill in the `None` placeholders below.

In [ ]:
def membrane_params(I_ext, Cm=1.0, gL=0.1, EL=-70.0):
    """
    Return (tau_m, V_inf) for the passive membrane.

    Parameters
    ----------
    I_ext : float  External current (µA/cm²)
    Cm    : float  Membrane capacitance (µF/cm²)
    gL    : float  Leak conductance (mS/cm²)
    EL    : float  Leak reversal potential (mV)

    Returns
    -------
    tau_m : float  Membrane time constant (ms)
    V_inf : float  Steady-state voltage (mV)
    """
    # --- YOUR CODE HERE ---
    tau_m = None   # Hint: Cm / gL  (units: µF/cm² ÷ mS/cm² = ms)
    V_inf = None   # Hint: EL + I_ext / gL
    # --- END ---
    return tau_m, V_inf


# Test with default parameters and I_ext = 1 µA/cm²
tau_m, V_inf = membrane_params(I_ext=1.0)
print(f'τ_m  = {tau_m} ms   (expected: 10.0 ms)')
print(f'V_∞  = {V_inf} mV   (expected: -60.0 mV)')

### Widget 1 — Interactive passive membrane

Use the sliders to explore how $I_{\text{ext}}$ and $g_L$ shape the membrane response.
The dashed lines mark $V_\infty$ and the time at which $V$ reaches $1 - e^{-1} \approx 63\%$ of the way to $V_\infty$.

In [ ]:
def plot_passive(I_ext=1.0, gL=0.1):
    Cm, EL, V0 = 1.0, -70.0, -70.0
    t, V = simulate_passive_membrane(I_ext=I_ext, Cm=Cm, gL=gL, EL=EL, V0=V0)
    tau_m, V_inf = membrane_params(I_ext=I_ext, Cm=Cm, gL=gL, EL=EL)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(t, V, color='steelblue', lw=1.8, label='V(t)')

    if tau_m is not None and V_inf is not None:
        ax.axhline(V_inf, color='tomato', ls='--', lw=1.2,
                   label=f'$V_\\infty$ = {V_inf:.1f} mV')
        ax.axvline(tau_m, color='orchid', ls=':', lw=1.2,
                   label=f'$\\tau_m$ = {tau_m:.1f} ms')

    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Membrane voltage (mV)')
    ax.set_title(f'Passive membrane  |  $I_{{ext}}$={I_ext} µA/cm²  $g_L$={gL} mS/cm²')
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.show()


interact(
    plot_passive,
    I_ext=widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1,
                              description='I_ext (µA/cm²)', style={'description_width': '130px'},
                              layout=widgets.Layout(width='420px')),
    gL=widgets.FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01,
                           description='g_L (mS/cm²)', style={'description_width': '130px'},
                           layout=widgets.Layout(width='420px')),
);

---
## Part 2 — Nernst Equilibrium Potential

### 2.1  Thermodynamic derivation

At rest, ions experience two competing forces:

| Force | Expression | Direction |
|---|---|---|
| **Chemical** (diffusion) | $RT \ln([X]_i / [X]_o)$ | down concentration gradient |
| **Electrical** | $zFV$ | depends on charge sign and $V$ |

**Nernst equilibrium** occurs when the net electrochemical potential is zero:

$$\mu_i^{\text{elec}} + \mu_i^{\text{chem}} = 0 \implies zF\,E_X + RT\ln\frac{[X]_i}{[X]_o} = 0$$

Solving for $E_X$:

$$\boxed{E_X = \frac{RT}{zF}\ln\frac{[X]_o}{[X]_i}}$$

At body temperature ($T = 310.15$ K), the prefactor $RT/F \approx 26.7$ mV.  
Multiplying by $\ln(10)/z$ gives the "58/z mV per decade" rule of thumb.

### 2.2  Typical mammalian values

| Ion | $[X]_i$ (mM) | $[X]_o$ (mM) | $z$ | $E_X$ (mV) |
|-----|-------------|-------------|-----|------------|
| K⁺  | 140 | 5   | +1 | ≈ −89 |
| Na⁺ | 15  | 145 | +1 | ≈ +62 |
| Cl⁻ | 10  | 110 | −1 | ≈ −87 |
| Ca²⁺| 0.0001 | 2 | +2 | ≈ +136 |

Note: $E_K$ is negative (K⁺ wants to leave, building inside-negative voltage); $E_{Na}$ is positive.

### Exercise 2.1 — Implement the Nernst equation

In [ ]:
def nernst(X_out, X_in, z, T=310.15):
    """
    Compute the Nernst equilibrium potential.

    Parameters
    ----------
    X_out : float  Extracellular concentration (mM)
    X_in  : float  Intracellular concentration (mM)
    z     : int    Ion valence (e.g. +1 for K⁺, -1 for Cl⁻, +2 for Ca²⁺)
    T     : float  Temperature (K)

    Returns
    -------
    E_X : float  Equilibrium potential (mV)
    """
    # --- YOUR CODE HERE ---
    # Hint 1: Use the global constants R (J/mol/K) and F (C/mol) defined in Setup
    # Hint 2: E_X = (R*T) / (z*F) * ln(X_out / X_in)
    # Hint 3: Convert result from Volts to millivolts (multiply by 1000)
    # Hint 4: Use np.log() for the natural logarithm
    pass
    # --- END ---


# Verification against physiological values
E_K  = nernst(CONC['K_out'],  CONC['K_in'],  z= 1)
E_Na = nernst(CONC['Na_out'], CONC['Na_in'], z= 1)
E_Cl = nernst(CONC['Cl_out'], CONC['Cl_in'], z=-1)
E_Ca = nernst(CONC['Ca_out'], CONC['Ca_in'], z= 2)

print('Nernst potentials (body temperature, 310.15 K):')
print(f'  E_K  = {E_K:.1f} mV   (expected ≈ -89 mV)')
print(f'  E_Na = {E_Na:.1f} mV   (expected ≈ +62 mV)')
print(f'  E_Cl = {E_Cl:.1f} mV   (expected ≈ -87 mV)')
print(f'  E_Ca = {E_Ca:.1f} mV   (expected ≈ +136 mV)')

### Exercise 2.2 — Short answer: Deriving the Nernst equation

**[Short answer]**  
In your own words (2–4 sentences), derive the Nernst equation by setting the sum of the chemical potential $\mu_\text{chem} = RT\ln([X]_i/[X]_o)$ and the electrical potential $\mu_\text{elec} = zFV$ equal to zero.  
Why must we take the ratio $[X]_o / [X]_i$ (not $[X]_i / [X]_o$) in the final expression?

*Your answer here.*

### Widget 2 — Nernst potential explorer

Adjust the extracellular and intracellular concentrations (on a log scale) and the valence to see how $E_X$ changes.

In [ ]:
def plot_nernst(log_X_out=np.log10(5.0), log_X_in=np.log10(140.0), z=1):
    X_out_val = 10**log_X_out
    X_in_val  = 10**log_X_in

    # Sweep over a range of X_out/X_in ratios
    ratios = np.logspace(-3, 3, 300)
    E_vals = [(R * 310.15) / (z * F) * np.log(r) * 1000 for r in ratios]

    E_current = nernst(X_out_val, X_in_val, z=z) if nernst(1, 1, 1) is not None else None

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.semilogx(ratios, E_vals, color='steelblue', lw=1.8,
                label=f'$E_X$ vs $[X]_o/[X]_i$  (z={z})')
    ax.axhline(0, color='gray', lw=0.8, ls='--')

    if E_current is not None:
        ratio_current = X_out_val / X_in_val
        ax.axvline(ratio_current, color='tomato', ls=':', lw=1.3)
        ax.scatter([ratio_current], [E_current], color='tomato', zorder=5,
                   label=f'Current: $E_X$ = {E_current:.1f} mV')

    ax.set_xlabel('$[X]_o / [X]_i$ (log scale)')
    ax.set_ylabel('Nernst potential $E_X$ (mV)')
    ax.set_title('Nernst equilibrium potential')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f'[X]_o = {X_out_val:.2f} mM,  [X]_i = {X_in_val:.2f} mM,  z = {z}')
    if E_current is not None:
        print(f'E_X = {E_current:.2f} mV')


interact(
    plot_nernst,
    log_X_out=widgets.FloatSlider(value=np.log10(5.0), min=-1.0, max=3.0, step=0.05,
                                  description='log₁₀[X]_o', style={'description_width': '110px'},
                                  layout=widgets.Layout(width='400px')),
    log_X_in =widgets.FloatSlider(value=np.log10(140.0), min=-1.0, max=3.0, step=0.05,
                                  description='log₁₀[X]_i', style={'description_width': '110px'},
                                  layout=widgets.Layout(width='400px')),
    z=widgets.IntSlider(value=1, min=-2, max=2, step=1,
                        description='Valence z', style={'description_width': '110px'},
                        layout=widgets.Layout(width='400px')),
);

---
## Part 3 — Non-Equilibrium: Goldman-Hodgkin-Katz Equation

### 3.1  Why Nernst is not enough

Nernst gives the equilibrium potential for **a single ion** when only that ion is permeable. At rest, the membrane is permeable to **multiple ions simultaneously** — and the membrane sits at a **non-equilibrium steady state** (there is net ion flow, balanced by the Na⁺/K⁺-ATPase pump).

### 3.2  The GHK voltage equation

Assuming constant electric field across the membrane and treating ions as independent, the steady-state membrane voltage $V_m$ satisfies:

$$\boxed{V_m = \frac{RT}{F} \ln \frac{P_{\text{K}}[\text{K}^+]_o + P_{\text{Na}}[\text{Na}^+]_o + P_{\text{Cl}}[\text{Cl}^-]_i}
                              {P_{\text{K}}[\text{K}^+]_i + P_{\text{Na}}[\text{Na}^+]_i + P_{\text{Cl}}[\text{Cl}^-]_o}}$$

where $P_X$ is the **permeability** of ion $X$ (units: cm/s, but only relative ratios matter).  
Notice that Cl⁻ concentrations appear **swapped** (inside appears in the numerator) because Cl⁻ carries negative charge.

### 3.3  Resting permeability ratios

At rest in a typical mammalian neuron:

$$\frac{P_{\text{Na}}}{P_{\text{K}}} \approx 0.04, \qquad \frac{P_{\text{Cl}}}{P_{\text{K}}} \approx 0.45$$

Because $P_K \gg P_{Na}$, the resting $V_m$ is pulled toward $E_K$ (≈ −89 mV) but not quite there — the small Na⁺ permeability depolarises it to roughly −68 mV.

### Exercise 3.1 — Implement the GHK voltage equation

In [ ]:
def GHK(P_Na, P_Cl, P_K=1.0, T=310.15,
        K_in=None,  K_out=None,
        Na_in=None, Na_out=None,
        Cl_in=None, Cl_out=None):
    """
    Goldman-Hodgkin-Katz voltage equation for K⁺, Na⁺, Cl⁻.

    Parameters
    ----------
    P_Na  : float  Na⁺ permeability relative to K⁺
    P_Cl  : float  Cl⁻ permeability relative to K⁺
    P_K   : float  K⁺ permeability (reference, default 1.0)
    T     : float  Temperature (K)
    K_in, K_out, Na_in, Na_out, Cl_in, Cl_out : float
            Ion concentrations (mM).  Default to CONC dict values when None.

    Returns
    -------
    V_m : float  Membrane potential (mV)
    """
    # Fill in defaults from the global CONC dictionary
    K_in  = CONC['K_in']  if K_in  is None else K_in
    K_out = CONC['K_out'] if K_out is None else K_out
    Na_in = CONC['Na_in'] if Na_in is None else Na_in
    Na_out= CONC['Na_out']if Na_out is None else Na_out
    Cl_in = CONC['Cl_in'] if Cl_in is None else Cl_in
    Cl_out= CONC['Cl_out']if Cl_out is None else Cl_out

    # --- YOUR CODE HERE ---
    # Hint 1: Build the numerator:   P_K*K_out  + P_Na*Na_out + P_Cl*Cl_in
    #         Build the denominator: P_K*K_in   + P_Na*Na_in  + P_Cl*Cl_out
    #         (Cl⁻ concentrations are swapped relative to the cations — why?)
    # Hint 2: V_m = (R*T/F) * ln(numerator / denominator)
    # Hint 3: Convert V → mV (multiply by 1000)
    pass
    # --- END ---


# Test: resting permeability ratios => V_m ≈ -68 mV
Vm = GHK(P_Na=0.04, P_Cl=0.45)
print(f'GHK resting V_m = {Vm:.2f} mV   (expected ≈ -68 mV)')

### Exercise 3.2 — Comparison table

Complete the cell below to print a tidy table comparing individual Nernst potentials with the GHK prediction.

In [ ]:
# --- YOUR CODE HERE ---
# Compute E_K, E_Na, E_Cl using your nernst() function (see Part 2 for z values)
# Compute Vm using GHK(P_Na=0.04, P_Cl=0.45)
# Print a neat table. Example format:
#
#   Ion      Equilibrium potential
#   ---      --------------------
#   E_K      -XX.X mV
#   E_Na     +XX.X mV
#   E_Cl     -XX.X mV
#   V_m(GHK) -XX.X mV

pass
# --- END ---

### Exercise 3.3 — Short answer: Why is $V_m$ closer to $E_K$?

**[Short answer]**  
Using the GHK equation as a guide, explain in 2–3 sentences why the resting membrane potential is much closer to $E_K$ than to $E_{Na}$, even though both ions are present.

*Your answer here.*

### Widget 3 — GHK permeability explorer

Move the sliders to change relative permeabilities. The plot shows $V_m$ vs $P_{Na}/P_K$ for the current $P_{Cl}/P_K$, with dashed Nernst limits.

In [ ]:
def plot_GHK(log_PNa_PK=np.log10(0.04), log_PCl_PK=np.log10(0.45)):
    P_Na_cur = 10**log_PNa_PK
    P_Cl_cur = 10**log_PCl_PK

    pna_range = np.logspace(-3, 1, 300)  # P_Na/P_K from 0.001 to 10
    Vm_curve  = [GHK(P_Na=p, P_Cl=P_Cl_cur) for p in pna_range]

    Vm_current = GHK(P_Na=P_Na_cur, P_Cl=P_Cl_cur) if GHK(0.04, 0.45) is not None else None
    E_K_val  = nernst(CONC['K_out'],  CONC['K_in'],  z=1)
    E_Na_val = nernst(CONC['Na_out'], CONC['Na_in'], z=1)

    fig, ax = plt.subplots(figsize=(7, 4))
    if Vm_curve[0] is not None:
        ax.semilogx(pna_range, Vm_curve, color='steelblue', lw=1.8,
                    label='$V_m$ (GHK)')

    if E_K_val is not None:
        ax.axhline(E_K_val,  color='seagreen', ls='--', lw=1.2,
                   label=f'$E_K$ = {E_K_val:.1f} mV')
    if E_Na_val is not None:
        ax.axhline(E_Na_val, color='tomato', ls='--', lw=1.2,
                   label=f'$E_{{Na}}$ = {E_Na_val:.1f} mV')

    if Vm_current is not None:
        ax.axvline(P_Na_cur, color='orchid', ls=':', lw=1.2)
        ax.scatter([P_Na_cur], [Vm_current], color='orchid', zorder=5,
                   label=f'Current: $V_m$ = {Vm_current:.1f} mV')

    ax.set_xlabel('$P_{Na} / P_K$ (log scale)')
    ax.set_ylabel('Membrane potential (mV)')
    ax.set_title(f'GHK  |  $P_{{Cl}}/P_K$ = {P_Cl_cur:.3f}')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


interact(
    plot_GHK,
    log_PNa_PK=widgets.FloatSlider(value=np.log10(0.04), min=-3.0, max=1.0, step=0.05,
                                   description='log₁₀(P_Na/P_K)', style={'description_width': '140px'},
                                   layout=widgets.Layout(width='440px')),
    log_PCl_PK=widgets.FloatSlider(value=np.log10(0.45), min=-3.0, max=1.0, step=0.05,
                                   description='log₁₀(P_Cl/P_K)', style={'description_width': '140px'},
                                   layout=widgets.Layout(width='440px')),
);

---
## Part 4 — Synaptic Input: Brief Current Pulses

### 4.1  Integration over the membrane time constant

A neuron does not respond to inputs instantaneously — it **integrates** them over a window set by $\tau_m$.

A simple model of synaptic excitation is a **rectangular current pulse** of amplitude $I_0$ turned on at $t_{\text{on}}$ and off at $t_{\text{off}}$:

$$I_{\text{ext}}(t) = I_0 \cdot \mathbf{1}_{[t_{\text{on}},\, t_{\text{off}}]}(t)$$

Inserting this into the passive membrane ODE gives a piece-wise exponential solution:
- **During the pulse** ($t_{\text{on}} \le t < t_{\text{off}}$): $V$ rises toward $V_\infty = E_L + I_0/g_L$
- **After the pulse** ($t \ge t_{\text{off}}$): $V$ decays back toward $E_L$

The **width of the voltage deflection** is broadened by $\tau_m$ — a key property that allows neurons to detect **coincident** inputs within a $\tau_m$ window.

### Exercise 4.1 — Simulate a synaptic current pulse

In [ ]:
def simulate_synaptic_input(I0=2.0, t_on=50.0, t_off=55.0,
                             T=200.0, dt=0.1,
                             Cm=1.0, gL=0.1, EL=-70.0, V0=-70.0):
    """
    Simulate a passive membrane receiving a rectangular current pulse.

    Parameters
    ----------
    I0    : float  Pulse amplitude (µA/cm²)
    t_on  : float  Pulse onset time (ms)
    t_off : float  Pulse offset time (ms)
    T, dt, Cm, gL, EL, V0 : same as simulate_passive_membrane

    Returns
    -------
    t : np.ndarray  Time vector (ms)
    V : np.ndarray  Voltage vector (mV)
    I : np.ndarray  Current vector (µA/cm²)  — for plotting
    """
    # --- YOUR CODE HERE ---
    # Hint 1: Build t array with np.arange(0, T, dt)
    # Hint 2: Build I array: I[k] = I0 if t_on <= t[k] < t_off, else 0.0
    #         np.where() or a simple boolean index both work well
    # Hint 3: Euler loop exactly as in Exercise 1.1, but use I[i-1] as I_ext
    pass
    # --- END ---


# Plot result
t, V, I = simulate_synaptic_input()

fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True,
                          gridspec_kw={'height_ratios': [1, 3]})

axes[0].plot(t, I, color='darkorange', lw=1.5)
axes[0].set_ylabel('$I_{ext}$ (µA/cm²)')
axes[0].set_title('Synaptic current pulse')

axes[1].plot(t, V, color='steelblue', lw=1.8)
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Membrane voltage (mV)')

plt.tight_layout()
plt.show()

### Exercise 4.2 — Short answer: pulse width vs $\tau_m$

**[Short answer]**  
Use Widget 4 below to experiment with very short (1 ms) and long (50 ms) pulse durations, and vary $g_L$ to change $\tau_m$.

1. What happens to the **peak voltage** when the pulse duration is much shorter than $\tau_m$?
2. What happens when the pulse duration is much longer than $\tau_m$?
3. In what sense does $\tau_m$ define a "coincidence detection window" for a neuron?

*Your answer here.*

### Widget 4 — Synaptic integration explorer

In [ ]:
def plot_synaptic(I0=2.0, gL=0.1, t_on=50.0, t_off=55.0):
    result = simulate_synaptic_input(I0=I0, t_on=t_on, t_off=t_off, gL=gL)
    if result is None:
        print('Implement simulate_synaptic_input() first.')
        return
    t, V, I = result
    tau_m = 1.0 / gL  # Cm = 1 µF/cm²

    fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True,
                              gridspec_kw={'height_ratios': [1, 3]})
    axes[0].plot(t, I, color='darkorange', lw=1.5)
    axes[0].set_ylabel('$I_{ext}$ (µA/cm²)')
    axes[0].set_title(f'$\\tau_m$ = {tau_m:.1f} ms  |  Pulse: [{t_on}–{t_off}] ms  |  '
                      f'Width = {t_off - t_on:.0f} ms')

    axes[1].plot(t, V, color='steelblue', lw=1.8)
    axes[1].axhline(-70.0, color='gray', ls='--', lw=0.8, label='$E_L$')
    axes[1].set_xlabel('Time (ms)')
    axes[1].set_ylabel('Membrane voltage (mV)')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.show()


interact(
    plot_synaptic,
    I0  =widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1,
                             description='I₀ (µA/cm²)', style={'description_width': '130px'},
                             layout=widgets.Layout(width='420px')),
    gL  =widgets.FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01,
                             description='g_L (mS/cm²)', style={'description_width': '130px'},
                             layout=widgets.Layout(width='420px')),
    t_on =widgets.FloatSlider(value=50.0, min=10.0, max=150.0, step=1.0,
                              description='t_on (ms)', style={'description_width': '130px'},
                              layout=widgets.Layout(width='420px')),
    t_off=widgets.FloatSlider(value=55.0, min=11.0, max=180.0, step=1.0,
                              description='t_off (ms)', style={'description_width': '130px'},
                              layout=widgets.Layout(width='420px')),
);

## Part 5 — From Biological Neuron to Artificial Neuron

The passive membrane equation you derived in Part 1 is not just a biophysics curiosity.
It is **the simplest artificial neuron** used in recurrent neural networks today.

Compare:

| Biological (leaky membrane) | Artificial (leaky integrator unit) |
|---|---|
| $C_m \frac{dV}{dt} = -g_L(V - E_L) + I_{\text{ext}}$ | $\tau \frac{dx}{dt} = -x + f\!\left(\mathbf{w}^T \mathbf{u} + b\right)$ |
| $V$ = membrane voltage | $x$ = unit activation |
| $\tau_m = C_m / g_L$ = time constant | $\tau$ = time constant |
| $I_{\text{ext}}$ = synaptic input current | $\mathbf{w}^T \mathbf{u}$ = weighted inputs |
| $E_L$ = resting potential | $b$ = bias |

The leaky integrator is the building block of **Liquid State Machines**, **continuous-time RNNs**, and **Neural ODEs** — all active frontiers of AI research.

Key insight: a biological neuron has **memory** (it integrates inputs over time $\sim \tau_m$).
A standard deep-learning neuron (perceptron) has **no memory** — it is purely instantaneous.
Adding dynamics (the $\tau$ term) is what makes recurrent networks powerful.

### Exercise 5.1 — The leaky integrator as an artificial neuron

We will drive the leaky integrator with a **time-varying input** $u(t)$ — a sequence of synaptic pulses — and observe how the unit *integrates* information over time.

This is exactly what a recurrent unit does when processing a sequence (speech, text, sensor data).

The equation (simplified, $E_L = 0$):

$$\tau \frac{dx}{dt} = -x + u(t)$$

where $u(t)$ is the input signal.

In [ ]:
def leaky_integrator(u_t, t, tau=10.0, x0=0.0):
    """
    Simulate a leaky integrator neuron driven by input u(t).

    Parameters
    ----------
    u_t : array  — input signal sampled at times t [same length as t]
    t   : array  — time vector [ms]
    tau : float  — time constant [ms]
    x0  : float  — initial state

    Returns
    -------
    x : array — unit activation over time
    """
    dt = t[1] - t[0]
    x = np.zeros(len(t))
    x[0] = x0

    for i in range(1, len(t)):
        # --- YOUR CODE HERE ---
        # Hint: dx/dt = (-x + u) / tau
        # Euler step: x[i] = x[i-1] + dt * dx_dt
        pass
        # --- END ---

    return x


# Build an input: 3 pulses of different amplitudes
t = np.arange(0, 300, 0.1)
u = np.zeros_like(t)
u[(t >= 20)  & (t < 30)]  = 2.0   # pulse 1
u[(t >= 80)  & (t < 100)] = 1.0   # pulse 2 (longer, weaker)
u[(t >= 160) & (t < 162)] = 5.0   # pulse 3 (brief, strong)

# --- YOUR CODE HERE ---
# Call leaky_integrator with tau=20 and tau=5, store results as x_slow and x_fast
x_slow = None
x_fast = None
# --- END ---

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(t, u, color='gray')
axes[0].set_ylabel('Input u(t)')
axes[0].set_title('Input signal')
axes[1].plot(t, x_slow, label='\u03c4 = 20 ms (slow)', color='steelblue')
axes[1].plot(t, x_fast, label='\u03c4 = 5 ms  (fast)', color='tomato')
axes[1].set_ylabel('Activation x(t)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title('Leaky integrator response')
axes[1].legend()
plt.tight_layout()
plt.show()

### Exercise 5.2 — Short answers

1. The slow integrator (large τ) and fast integrator (small τ) respond differently to the three input pulses. Which one "remembers" pulse 1 when pulse 2 arrives? What does this mean for sequence processing in AI?

2. The brief, strong pulse 3 produces a large transient in the fast integrator but barely moves the slow one. Relate this to the concept of a **temporal receptive field** in sensory neuroscience.

3. A standard deep-learning neuron (perceptron) corresponds to τ → 0 in this model. What does that mean for its ability to process time-varying inputs?

> **Your answers (double-click to edit):**
>
> 1. ...
> 2. ...
> 3. ...

### Widget 5 — Leaky integrator explorer

In [ ]:
def plot_leaky(tau_slow, tau_fast, pulse1_amp, pulse2_dur):
    t = np.arange(0, 300, 0.1)
    u = np.zeros_like(t)
    u[(t >= 20)  & (t < 30)]               = pulse1_amp
    u[(t >= 80)  & (t < 80 + pulse2_dur)]  = 1.0
    u[(t >= 160) & (t < 162)]              = 5.0

    x_slow = leaky_integrator(u, t, tau=tau_slow)
    x_fast = leaky_integrator(u, t, tau=tau_fast)

    fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    axes[0].fill_between(t, u, alpha=0.4, color='gray')
    axes[0].plot(t, u, color='gray', lw=1)
    axes[0].set_ylabel('u(t)')
    axes[1].plot(t, x_slow, lw=2, label=f'\u03c4 = {tau_slow} ms', color='steelblue')
    axes[1].plot(t, x_fast, lw=2, label=f'\u03c4 = {tau_fast} ms', color='tomato')
    axes[1].set_ylabel('x(t)')
    axes[1].set_xlabel('Time (ms)')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

widgets.interact(
    plot_leaky,
    tau_slow =widgets.FloatSlider(value=20, min=5,  max=100, step=5,   description='\u03c4 slow (ms)'),
    tau_fast =widgets.FloatSlider(value=5,  min=1,  max=30,  step=1,   description='\u03c4 fast (ms)'),
    pulse1_amp=widgets.FloatSlider(value=2, min=0.5, max=8,  step=0.5, description='Pulse 1 amp'),
    pulse2_dur=widgets.FloatSlider(value=20,min=2,  max=80,  step=2,   description='Pulse 2 dur (ms)')
);

---
## Summary — Key Equations

| Quantity | Symbol | Expression | Units |
|---|---|---|---|
| Membrane time constant | $\tau_m$ | $C_m / g_L$ | ms |
| Steady-state voltage | $V_\infty$ | $E_L + I_{\text{ext}} / g_L$ | mV |
| Passive membrane ODE | — | $C_m \dot{V} = -g_L(V - E_L) + I_{\text{ext}}$ | µA/cm² |
| Nernst potential | $E_X$ | $\dfrac{RT}{zF}\ln\dfrac{[X]_o}{[X]_i}$ | mV |
| GHK voltage | $V_m$ | $\dfrac{RT}{F}\ln\dfrac{P_K[K]_o + P_{Na}[Na]_o + P_{Cl}[Cl]_i}{P_K[K]_i + P_{Na}[Na]_i + P_{Cl}[Cl]_o}$ | mV |
| Resting $P_{Na}/P_K$ | — | ≈ 0.04 | — |
| Thermal voltage ($T$ = 310 K) | $RT/F$ | ≈ 26.7 mV | mV |

---

### Further reading

- Dayan & Abbott (2001), *Theoretical Neuroscience*, Chapter 5 — Conductance-Based Models.
- Hille (2001), *Ion Channels of Excitable Membranes*, Chapter 14 — GHK theory.
- Koch (1999), *Biophysics of Computation*, Chapter 2 — Cable Theory.

---
*End of Lab 01*